# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display available record sets with their @id, name, and field information
record_sets = list(dataset.metadata.record_sets)
print("Available Record Sets:")
for rset in record_sets:
    print(f'  @id: {rset.id}')
    print(f'    name: {rset.name}')
    # Show fields and columns
    print("    Fields & Columns:")
    for field in rset.fields:
        print(f'      Field @id: {field.id} | name: {getattr(field, "name", "N/A")}')
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# -- Set up to extract data from all record sets found --
all_record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'Loaded {len(df)} records for record set: {record_set_id}')

# Pick the main record set (the first one, or choose by description)
main_record_set_id = all_record_set_ids[0] if all_record_set_ids else None
if main_record_set_id:
    print(f'Example columns for {main_record_set_id}:')
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# -- EDA on a numeric field: choose a numeric field from the columns --
import numpy as np

eda_record_set_id = main_record_set_id
df = dataframes[eda_record_set_id].copy() if eda_record_set_id else None

# Inspect columns to choose a numeric_field
print(f"Columns available in {eda_record_set_id}:")
if df is not None:
    print(df.columns.tolist())

# Assume common mass spec field: e.g. 'MW', 'abundance', 'PeptideCount', substitute as appropriate
# Try typical numeric columns (first checking for these common ones)
candidate_numeric_fields = ['MW', 'molecular_weight', 'abundance', 'Abundance', 'coverage', 'Coverage', 'PeptideCount', 'peptide_count']
numeric_field = None
for col in df.columns:
    if col in candidate_numeric_fields or np.issubdtype(df[col].dtype, np.number):
        numeric_field = col
        break

if numeric_field is None:
    print("No suitable numeric field found. Please check the columns above.")
else:
    # Drop NA and convert to numeric if needed
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric column
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a common categorical field
    candidate_group_fields = ['Protein', 'Gene', 'gene', 'sample', 'Sample', 'description', 'Description']
    group_field = None
    for col in df.columns:
        if col in candidate_group_fields:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped (mean {numeric_field}) by {group_field}:")
        display(grouped_df.head())
    else:
        print("No obvious categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we found both numeric_field and group_field, do a boxplot
    if group_field and df[group_field].nunique() < 20:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded using the Croissant schema and the `mlcroissant` library, with all key entities referenced by their `@id`.
- Multiple record sets and fields were explored dynamically; main record set data was extracted to pandas DataFrames.
- Simple exploratory data analysis (filtering, normalization, grouping) and basic visualizations were performed.
- For custom analyses, refer to field and record set `@id`s, and adjust the EDA/visualization cells appropriately.

For more details or advanced use (e.g., handling nested/linked data relations), consult the relevant Croissant documentation and the `mlcroissant` API.